# DDOS Identification - example usage

## Preprocessing

### Load dataset

In [ ]:
import pandas as pd
import numpy as np

# Loading logic
df = pd.read_csv("data.csv")

# Uncomment to view the first 5 rows
# df.head(5)

# Done
print("File loaded")

### IP address segmenting 

In [ ]:
# Convert IP strings into octets efficiently using NumPy
def ip_to_octets(ip_series, prefix):
    octets = np.stack(ip_series.str.split('.').apply(lambda x: list(map(int, x))))
    return pd.DataFrame(octets, columns=[f"{prefix} Octet{i+1}" for i in range(4)])

# Apply the optimized function
src_octets = ip_to_octets(df["Src IP"], "Src IP")
dst_octets = ip_to_octets(df["Dst IP"], "Dst IP")

# Concatenate with original DataFrame
df = pd.concat([df, src_octets, dst_octets], axis=1).drop(columns=["Src IP", "Dst IP"])

# Delete unused vars
del src_octets
del dst_octets

# Uncomement below to view the first 5 rows
# df.head(5)

# Done
print("IP segmentation Done")

### Drop unneeded columns

In [ ]:
df.drop(columns=["Unnamed: 0", "Flow ID"], inplace=True)

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

### Load preprocessing models

In [ ]:
imputer = joblib.load("imputer.joblib")
scaler = joblib.load("scaler.joblib")

### Impute missing values

In [ ]:
df[numerical_cols] = pd.DataFrame(imputer.transform(df[numerical_cols]), columns=numerical_cols, index=df.index)  # DO NOT remove the index=xxx attribute, else NaN happens

# Uncomement below to view the first 5 rows
# df.head(5)

# done
print("Imputing done")

### Scale dataset

In [ ]:
df[numerical_cols] = pd.DataFrame(scaler.transform(df[numerical_cols]), columns=numerical_cols, index=df.index)

# Uncomement below to view the first 5 rows
# df.head(5)

# done
print("Scaling done")

**End of preprocessing**

## Model loading

In [ ]:
import json
import torch
import joblib
from sklearn.linear_model import LogisticRegression

# Model files
log_reg_file = "*.joblib"
autoencoder_params_file = "*.pth" # state_dict(); i.e. model
autoencoder_hyperparams_file = "*.json" # 
autoencoder_errors_file = "*.json"
lstm_params_file = "*.pth"
lstm_hyperparams_file = "*.json"

# Loading model hyperparams
with open(autoencoder_hyperparams_file, "r") as f:
    autoencoder_hyperparams = json.load(f)
with open(lstm_hyperparams_file, "r") as f:
    lstm_hyperparams = json.load(f)
with open(autoencoder_errors_file, "r") as f:
    autoencoder_errors = json.load(f)
    

# Initialize model with loaded hyperparams
autoencoder = Autoencoder(**autoencoder_hyperparams)
lstm = LSTM(**lstm_hyperparams)

# Load model weights
autoencoder.load_state_dict(torch.load(autoencoder_params_file))
lstm.load_state_dict(torch.load(lstm_params_file))

# Set torch models to evaluate mode (i.e. predictions mode)
autoencoder.eval()
lstm.eval()

# Load logistic regression
log_reg = joblib.load(log_reg_file)

## Predictions pipeline

### Data Definitions

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
X_numpy = df.values
X_log_reg = df[numerical_cols].values
X_autoencoder = torch.FloatTensor(df[numerical_cols].values)
X_lstm = torch.FloatTensor(df[numerical_cols].values)

### Logistic Regression Predictions

In [ ]:
# Get predictions
preds_log_reg = log_reg.predict(X_log_reg)

# Get indices of predicted DDoS
logistic_ddos_indices = np.where(logistic_preds == 1)[0]

### Autoencoder predictions

In [ ]:
# Set to evaluate mode
autoencoder.eval()

with torch.no_grad():
    reconstructed = autoencoder(X_tensor)  # Autoencoder output
    reconstruction_error = torch.mean((X_tensor - reconstructed) ** 2, dim=1)  # Compute MSE per sample

threshold = autoencoder_errors.threshold
autoencoder_preds = (reconstruction_error > threshold).cpu().numpy().astype(int)  # Convert to binary

autoencoder_ddos_indices = np.where(autoencoder_preds == 1)[0]  # Get indices of DDoS


### Combine Layer 1 suspected

In [ ]:
ddos_indices_combined = np.union1d(logistic_ddos_indices, autoencoder_ddos_indices)  # Merge indices
X_ddos = X_numpy[ddos_indices_combined]  # Extract DDoS samples
X_ddos_tensor = torch.tensor(X_ddos[numerical_cols]).to(device)  # Convert for LSTM

### LSTM predictions (i.e. Layer 2 predictions)

In [ ]:
lstm.eval()
with torch.no_grad():
    lstm_outputs = lstm(X_ddos_tensor)
        _, lstm_preds = torch.max(lstm_outputs, 1)  # Convert to class predictions

lstm_ddos_indices = ddos_indices_combined[np.where(lstm_preds.cpu().numpy() == 1)[0]]  # Get final DDoS indices

### Get additional info

In [ ]:
# Get additional details from the original dataset
ddos_data_final = original_dataset.iloc[lstm_ddos_indices]  # Extract corresponding rows

# Select key fields like source IP, destination IP, and ports
key_fields = [ 
    'Flow ID',
    'Src IP',
    'Src Port',
    'Dst IP',
    'Dst Port',
    'Protocol',
    'Timestamp'
]
ddos_key_info = ddos_data_final[key_fields]

# Get first 10
ddos_key_info.head(10)

# Store
ddos_key_info.to_csv("ddos_key_info.csv")